## 1장 1강 : LLM 애플리케이션의 입력과 출력 구조

### 의존 패키지 설치
```
uv add ipykernel langchain langchain-core langchain-ollama python-dotenv
```

### 3. LCEL 파이프라인 실습

#### 3.1 환경 변수 로드 및 패키지 불러오기

In [17]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama

#### 3.2 ChatPromptTemplate으로 입력 템플릿 조립하기

시스템 역할과 사용자 질문 변수({user_question})를 담은 템플릿 생성<br>
system: "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."<br>
user: "{user_question}"

In [ ]:
# system: 역할, 상황, 제한 조건, 예시 -> 시스템 메세지, SystemMessage(..)
# user: 사용자의 질의 : HumanMessage(..)
# asistant: AI의 답변 : AIMessage(..)

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요."),
    ("user", "{user_question}")
])

템플릿에 텍스트용 질문 데이터를 주입하여 결과를 확인

user_question: "프로그래밍에서 '변수'가 무엇인가요?"

In [19]:
sample_prompt = prompt_template.invoke({
    "user_question": "프로그래밍에서 '변수'가 무엇인가요?"
})

sample_prompt

ChatPromptValue(messages=[SystemMessage(content='당신은 컴퓨터 기초 개념을 일상적인 사물에 빗대어 설명하는 교육 전문가입니다. 2문장 이내로 친절하게 설명하세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content="프로그래밍에서 '변수'가 무엇인가요?", additional_kwargs={}, response_metadata={})])

#### 3.3 ChatOllama로 mistral 모델 호출하기

ChatOllama 모델 인스턴스 생성

In [20]:
model = ChatOllama(
    model = "mistral",
    base_url = "http://localhost:11434" # Ollama 서버 주소
)

model

ChatOllama(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, model='mistral', base_url='http://localhost:11434')

앞서 만든 sample_prompt를 모델에 직접 전달하여 실행

In [21]:
response = model.invoke(sample_prompt)

response

AIMessage(content=" 변수는 프로그램 내에서 데이터를 저장하고 변경할 수 있는 상자라고 할 수 있습니다. 예를 들어, 우리가 주택을 살때, 주소는 특정 집에 대한 '변수'와 같습니다. 이 주소를 변경하면, 집을 찾기 위한 방법이 달라지는 것과 같이, 프로그램에서도 변수의 값을 변경하면, 그 변수에 저장된 데이터가 달라집니다.", additional_kwargs={}, response_metadata={'model': 'mistral', 'created_at': '2026-09-09T06:08:47.0284534Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35456947500, 'load_duration': 8901428700, 'prompt_eval_count': 106, 'prompt_eval_duration': 4221586000, 'eval_count': 173, 'eval_duration': 22302495000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'}, id='lc_run--01a084c7-eb32-7900-91b5-e5c412c5c48b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 106, 'output_tokens': 173, 'total_tokens': 279})

#### 3.4 StrOutputParser로 순수 텍스트만 추출하기

문자열 출력 파서 생성

In [22]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

raw_response 객체에서 순수 텍스트만 추출

In [23]:
text = parser.invoke(response)

text

" 변수는 프로그램 내에서 데이터를 저장하고 변경할 수 있는 상자라고 할 수 있습니다. 예를 들어, 우리가 주택을 살때, 주소는 특정 집에 대한 '변수'와 같습니다. 이 주소를 변경하면, 집을 찾기 위한 방법이 달라지는 것과 같이, 프로그램에서도 변수의 값을 변경하면, 그 변수에 저장된 데이터가 달라집니다."

#### 3.5 LCEL 파이프(|) 연산자로 완전한 체인 결합 및 실행하기

파이프(|) 연산자를 사용해 3개 컴포넌트를 하나의 파이프라인으로 연결

In [24]:
chain = prompt_template | model | parser

새로운 질문으로 파이프라인 전체 실행

In [25]:
res = chain.invoke({
    "user_question": "클래스(Class)에 대해 알기 쉬운 비유를 통해 설명하세요"
})

res

' 클래스(Class)는 실세계에서의 집합체와 같습니다. 예를 들어, 학교에 여러 학생들이 있습니다. 학생들은 각자 다른 이름, 나이, 성별 등의 특성이 있지만, 같은 학교에 속하는 공통된 특성이 있습니다. 이러한 공통된 특성을 정의한 것이 바로 클래스입니다. 여기서 학생들은 클래스(학생)의 인스턴스(개인)입니다. 이처럼 클래스는 공통된 속성과 기능을 정의하고, 그것을 토대로 특정 객체를 만드는 데 사용됩니다.'

In [26]:
res1 = chain.invoke({
    "user_question": "오늘은 무슨년 몇월 며칠이야"
})

res1

' 오늘은 20XX년 XX월 XX일입니다! 이렇게 해서 년도, 월, 일을 알아볼 수 있으니까, 컴퓨터에서 날짜를 저장하는 것과 같아요. 예를 들어, 파일을 만들 때 파일 이름에 날짜를 붙이는 것도 같아요!'

### 4. 프롬프트 엔지니어링
#### 4.1 프롬프트 엔지니어링의 3대 핵심 역할
- 목표 명확화
- 제약 조건 부여
- 출력 규격 표준화

#### 4.2 AI 성능 확장 4단계 비교
- Prompt Engineering
- RAG (검색 증강 생성)
- Fine-tuning (미세 조정)
- AI Agent (에이전트)